In [31]:
import os
import ast
import pprint
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool
import pandas as pd

In [32]:

league_prefix = "laliga"
seasons = ["2526", "2425", "2324"]

# delete pl_tmp.db if it exists
if os.path.exists(f"{league_prefix}_tmp.db"):
    os.remove(f"{league_prefix}_tmp.db")


def get_engine_for_pl_db(seasons):
    """Pull sql file, populate in-memory database, and create engine."""

    connection = sqlite3.connect(
        f"{league_prefix}_tmp.db", check_same_thread=False)
    for season in seasons:
        df = pd.read_csv(f"./{league_prefix}_data/season-{season}.csv")
        df.to_sql(f"{league_prefix}_matches_{season}", connection,
                if_exists='replace', index=False)

    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


engine = get_engine_for_pl_db(seasons)
db = SQLDatabase(engine)
print(db.run("SELECT name FROM sqlite_master WHERE type='table';"))

[('laliga_matches_2526',), ('laliga_matches_2425',), ('laliga_matches_2324',)]


In [33]:

for season in seasons:
    points_table = pd.DataFrame(columns=["Team",
                                     "Matches_Played",
                                     "Points",
                                     "Goals_For",
                                     "Goals_Against",
                                     "Goal_Difference",
                                     "Shots_On_Target_For",
                                     "Shots_On_Target_Against",
                                     "Corners_For",
                                     "Corners_Against",
                                     "Red_Cards_For",
                                     "Red_Cards_Against",
                                     "Yellow_Cards_For",
                                     "Yellow_Cards_Against"])
    teams = db.run(f"SELECT DISTINCT HomeTeam FROM {league_prefix}_matches_{season};")
    print(teams)
    teams = teams.split(",")
    print("Teams in the database:")
    team_names = []
    for i, team in enumerate(teams):
        # print(team)
        if len(team.split(",")[0].split("(")) < 2:
            continue
        else:
            team_name = team.split(",")[0].split("(")[1].split(")")[0]
            print(team_name)
            if team_name[0] == '"':
                team_name = team_name.strip().split('"')[1]
            else:
                team_name = team_name.strip().split("'")[1]
            team_names.append(team_name)
    print(team_names, len(team_names))
    for team_name in team_names:
        resp = db.run(
            f"""
            SELECT HomeTeam, COUNT(HomeTeam) AS Matches_Played,  SUM(FTHG) AS Goals_For, SUM(FTAG) AS Goals_Against, SUM(CASE WHEN FTR = 'H' THEN 3 WHEN FTR = 'D' THEN 1 ELSE 0 END) AS Points, SUM(HST) AS Shots_On_Target_For, SUM(AST) AS Shots_On_Target_Against, SUM(HC) AS Corners_For, SUM(AC) AS Corners_Against, SUM(HR) AS Red_Cards_For, SUM(AR) AS Red_Cards_Against, SUM(HY) AS Yellow_Cards_For, SUM(AY) AS Yellow_Cards_Against
            FROM {league_prefix}_matches_{season} 
            WHERE HomeTeam LIKE "%{team_name}%"
            """,
            include_columns=True
        )
        resp = ast.literal_eval(resp)[0]
        if resp:
            points_table.loc[len(points_table)] = [
                team_name,
                resp["Matches_Played"],
                resp["Points"],
                resp["Goals_For"],
                resp["Goals_Against"],
                resp["Goals_For"] - resp["Goals_Against"],
                resp["Shots_On_Target_For"],
                resp["Shots_On_Target_Against"],
                resp["Corners_For"],
                resp["Corners_Against"],
                resp["Red_Cards_For"],
                resp["Red_Cards_Against"],
                resp["Yellow_Cards_For"],
                resp["Yellow_Cards_Against"]
            ]

        resp = db.run(
            f"""
            SELECT AwayTeam, COUNT(AwayTeam) AS Matches_Played,  SUM(FTAG) AS Goals_For, SUM(FTHG) AS Goals_Against, SUM(CASE WHEN FTR = 'A' THEN 3 WHEN FTR = 'D' THEN 1 ELSE 0 END) AS Points, SUM(AST) AS Shots_On_Target_For, SUM(HST) AS Shots_On_Target_Against, SUM(AC) AS Corners_For, SUM(HC) AS Corners_Against, SUM(AR) AS Red_Cards_For, SUM(HR) AS Red_Cards_Against, SUM(AY) AS Yellow_Cards_For, SUM(HY) AS Yellow_Cards_Against
            FROM {league_prefix}_matches_{season} 
            WHERE AwayTeam LIKE "%{team_name}%"
            """,
            include_columns=True
        )
        resp = ast.literal_eval(resp)[0]
        print(resp)
        if resp:
            points_table.iloc[len(points_table) - 1] = [
                team_name,
                points_table.iloc[len(points_table) -
                                1]["Matches_Played"] + resp["Matches_Played"],
                points_table.iloc[len(points_table) -
                                1]["Points"] + resp["Points"],
                points_table.iloc[len(points_table) -
                                1]["Goals_For"] + resp["Goals_For"],
                points_table.iloc[len(points_table) -
                                1]["Goals_Against"] + resp["Goals_Against"],
                points_table.iloc[len(points_table) - 1]["Goal_Difference"] +
                resp["Goals_For"] - resp["Goals_Against"],
                points_table.iloc[len(points_table) -
                                1]["Shots_On_Target_For"] + resp["Shots_On_Target_For"],
                points_table.iloc[len(
                    points_table) - 1]["Shots_On_Target_Against"] + resp["Shots_On_Target_Against"],
                points_table.iloc[len(points_table) -
                                    1]["Corners_For"] + resp["Corners_For"],
                points_table.iloc[len(points_table) -
                                    1]["Corners_Against"] + resp["Corners_Against"],
                points_table.iloc[len(points_table) -
                                    1]["Red_Cards_For"] + resp["Red_Cards_For"],
                points_table.iloc[len(
                    points_table) - 1]["Red_Cards_Against"] + resp["Red_Cards_Against"],
                points_table.iloc[len(
                    points_table) - 1]["Yellow_Cards_For"] + resp["Yellow_Cards_For"],
                points_table.iloc[len(
                    points_table) - 1]["Yellow_Cards_Against"] + resp["Yellow_Cards_Against"]
            ]
    points_table = points_table.sort_values(
        by="Points", ascending=False).reset_index(drop=True)
    # Add a position column based on the sorted points
    points_table.insert(0, "Position", range(1, len(points_table) + 1))
    print(points_table)
    points_table.to_csv(f"{league_prefix}_{season}_points_table.csv", index=False)


[('Girona',), ('Villarreal',), ('Mallorca',), ('Alaves',), ('Valencia',), ('Celta',), ('Ath Bilbao',), ('Espanol',), ('Elche',), ('Real Madrid',), ('Betis',), ('Ath Madrid',), ('Levante',), ('Osasuna',), ('Sociedad',), ('Oviedo',), ('Sevilla',), ('Vallecano',), ('Getafe',), ('Barcelona',)]
Teams in the database:
'Girona'
'Villarreal'
'Mallorca'
'Alaves'
'Valencia'
'Celta'
'Ath Bilbao'
'Espanol'
'Elche'
'Real Madrid'
'Betis'
'Ath Madrid'
'Levante'
'Osasuna'
'Sociedad'
'Oviedo'
'Sevilla'
'Vallecano'
'Getafe'
'Barcelona'
['Girona', 'Villarreal', 'Mallorca', 'Alaves', 'Valencia', 'Celta', 'Ath Bilbao', 'Espanol', 'Elche', 'Real Madrid', 'Betis', 'Ath Madrid', 'Levante', 'Osasuna', 'Sociedad', 'Oviedo', 'Sevilla', 'Vallecano', 'Getafe', 'Barcelona'] 20
{'AwayTeam': 'Girona', 'Matches_Played': 18, 'Goals_For': 18, 'Goals_Against': 27, 'Points': 17, 'Shots_On_Target_For': 68, 'Shots_On_Target_Against': 98, 'Corners_For': 60, 'Corners_Against': 114, 'Red_Cards_For': 0, 'Red_Cards_Against': 2, 

In [34]:
# 1. Connect to the SQLite database (this creates the file if it doesn't exist)
conn = sqlite3.connect(f'{league_prefix}.db')

# 2. Load the CSV file into a Pandas DataFrame
updated_columns = ["Date", "HomeTeam", "AwayTeam", "Full_Time_Home_Team_Goals", "Full_Time_Away_Team_Goals",
"Full_Time_Result", "Half_Time_Home_Team_Goals", "Half_Time_Away_Team_Goals", "Half_Time_Result", "Match_Referee",
"Home_Team_Shots", "Away_Team_Shots", "Home_Team_Shots_on_Target", "Away_Team_Shots_on_Target", "Home_Team_Fouls_Committed", "Away_Team_Fouls_Committed", "Home_Team_Corners", "Away_Team_Corners", "Home_Team_Yellow_Cards", "Away_Team_Yellow_Cards", "Home_Team_Red_Cards", "Away_Team_Red_Cards"]

df = pd.read_csv(f'./{league_prefix}_data/season-2526.csv')
# update column names
df.columns = updated_columns
df_2425 = pd.read_csv(f'./{league_prefix}_data/season-2425.csv')
df_2425.columns = updated_columns
df_2324 = pd.read_csv(f'./{league_prefix}_data/season-2324.csv')
df_2324.columns = updated_columns
pt_table_df = pd.read_csv(f'{league_prefix}_2526_points_table.csv')
pt_table_2425_df = pd.read_csv(f'{league_prefix}_2425_points_table.csv')
pt_table_2324_df = pd.read_csv(f'{league_prefix}_2324_points_table.csv')
# 3. Write the data to an SQLite table
# if_exists options: 'fail', 'replace', or 'append'
df.to_sql(f'{league_prefix}_matches_2526', conn, if_exists='replace', index=False)
df_2425.to_sql(f'{league_prefix}_matches_2425', conn, if_exists='replace', index=False)
df_2324.to_sql(f'{league_prefix}_matches_2324', conn, if_exists='replace', index=False)
pt_table_df.to_sql(f'{league_prefix}_table_2526', conn, if_exists='replace', index=False)
pt_table_2425_df.to_sql(f'{league_prefix}_table_2425', conn, if_exists='replace', index=False)
pt_table_2324_df.to_sql(f'{league_prefix}_table_2324', conn, if_exists='replace', index=False)

# 4. Close the connection
conn.close()

print("CSV loaded successfully into db!")

CSV loaded successfully into db!


In [39]:
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

league_prefix = "pl"
def get_engine_for_pl_db():
    """Pull sql file, populate in-memory database, and create engine."""


    connection = sqlite3.connect(f"{league_prefix}.db", check_same_thread=False)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_engine_for_pl_db()
db = SQLDatabase(engine)
# Print the tables in the database to verify connection
print(db.run("SELECT name FROM sqlite_master WHERE type='table';"))

[('pl_matches_2526',), ('pl_matches_2425',), ('pl_matches_2324',), ('pl_table_2526',), ('pl_table_2425',), ('pl_table_2324',)]


In [41]:
db.run("SELECT DISTINCT Team FROM pl_table_2526")

'[(\'Arsenal\',), (\'Man City\',), (\'Man United\',), (\'Liverpool\',), (\'Aston Villa\',), (\'Bournemouth\',), (\'Brighton\',), (\'Brentford\',), (\'Chelsea\',), (\'Everton\',), (\'Sunderland\',), (\'Fulham\',), (\'Newcastle\',), (\'Leeds\',), (\'Crystal Palace\',), ("Nott\'m Forest",), (\'Tottenham\',), (\'West Ham\',), (\'Burnley\',), (\'Wolves\',)]'